# 01 Event timeline

**Goal:** put everything that happened on the laser (jobs, alarms, maintenance, inspections) into **one table, sorted by time**.
This is the event layer of the knowledge graph, and the Python version of what you did by hand in Excel.

**How to use this notebook:** run one block at a time with **Shift + Enter**. Read the text above each block first.
Every block says which **EDA finding** it uses, so you can see why stage 2 mattered.

Nothing here changes the files in `datasets/`. The result is saved to `data/processed/`, which git ignores.

## 0. Setup
`pandas` is Python's version of Excel: a table is called a **DataFrame**.
`Path` just builds file paths that work on Windows and Mac.

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("..") / "datasets"          # this notebook lives in notebooks/, the data one folder up
OUT = Path("..") / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
print("pandas", pd.__version__)
print("datasets folder found:", DATA.exists())

pandas 3.0.3
datasets folder found: True


## 1. Job log
**EDA finding used:** the file uses `;` as separator, `,` as decimal mark and `dd.mm.yyyy` dates (German export). It has 4 exact duplicate rows.

In Power Query you fixed this with *Using Locale → German*. Here it's two settings: `sep=";"` and `decimal=","`.

In [2]:
jobs = pd.read_csv(DATA / "machine" / "job_log.csv", sep=";", decimal=",")
jobs["StartTime"] = pd.to_datetime(jobs["StartTime"], format="%d.%m.%Y %H:%M:%S")
jobs["EndTime"] = pd.to_datetime(jobs["EndTime"], format="%d.%m.%Y %H:%M:%S")

print("rows before:", len(jobs))
jobs = jobs.drop_duplicates()            # exact duplicates only, like Remove Duplicates on all columns
print("rows after :", len(jobs), "| unique JobIDs:", jobs["JobID"].nunique())
jobs.head(3)

rows before: 531
rows after : 527 | unique JobIDs: 527


,JobID,StartTime,EndTime,ProgramName,Material,Thickness_mm,CuttingGas,LaserPower_W,FeedRate_mm_min,GasPressureSet_bar,FocusPosition_mm,NozzleDiameter_mm,Standoff_mm,BevelAngle_deg,NozzleOperatingHours,PartsOnSheet,Status,Operator
0,JOB-00001,2026-09-01 06:25:00,2026-09-01 07:35:00,S235JR_12mm_BEV,S235JR,12,O2,6000,1520,0.6,4.0,1.4,1.2,22.5,0.0,42,Completed,MR
1,JOB-00002,2026-09-01 07:46:00,2026-09-01 09:15:00,S235JR_12mm,S235JR,12,O2,6000,1540,0.6,4.5,1.4,1.0,0.0,1.3,12,Completed,MR
2,JOB-00003,2026-09-01 09:37:00,2026-09-01 10:51:00,14301_6mm,1.4301,6,N2,6000,3140,18.2,-3.5,2.3,0.8,0.0,3.0,60,Completed,KT


## 2. Inspection records
**EDA findings used:**
- Some job IDs were typed like `job00123`, so we keep the last 5 digits and rebuild `JOB-00123`.
- Defect names are free text, with 15 spellings for 5 defects. This is your **Mapping** sheet as a Python dictionary.
- NOK with no defect written becomes `unknown`.

In [3]:
insp = pd.read_csv(DATA / "quality" / "inspection_records.csv")
insp["job_id"] = "JOB-" + insp["job_ref"].str[-5:]
insp["inspected_at"] = pd.to_datetime(insp["inspected_at"], format="ISO8601")

MAPPING = {
    "dross": "dross", "dross/burr": "dross", "burr underside": "dross", "slag at bottom edge": "dross",
    "rough cut": "rough_cut", "rough edge": "rough_cut", "striations": "rough_cut", "heavy striations": "rough_cut",
    "incomplete cut": "incomplete_cut", "not cut through": "incomplete_cut", "parts stuck in skeleton": "incomplete_cut",
    "bevel angle off": "bevel_deviation", "angle deviation": "bevel_deviation", "bevel nok": "bevel_deviation",
    "burnt edge": "burnt_edge",
}

def to_defect_type(row):
    if row["result"] == "OK":
        return "ok"
    if pd.isna(row["defect"]):
        return "unknown"
    return MAPPING.get(row["defect"].strip().lower(), "CHECK")   # CHECK = a spelling the mapping doesn't know

insp["defect_type"] = insp.apply(to_defect_type, axis=1)
print(insp["defect_type"].value_counts())
print("IDs that don't match any job:", (~insp["job_id"].isin(jobs["JobID"])).sum())

defect_type
ok                 250
dross               51
incomplete_cut      51
rough_cut           37
unknown             14
bevel_deviation     12
burnt_edge           1
Name: count, dtype: int64
IDs that don't match any job: 0


## 3. Alarm log
**EDA finding used:** the first 2 lines start with `#` (comments), and the columns are separated by `|` with spaces around it.

In Power Query this took 6 steps. Here it's `comment="#"` and `sep="|"`, then trim the spaces.

In [4]:
alarms = pd.read_csv(DATA / "machine" / "alarm_log.txt", sep="|", comment="#", header=None,
                     names=["timestamp", "code", "level", "message"])
for col in ["level", "message"]:
    alarms[col] = alarms[col].str.strip()
alarms["timestamp"] = pd.to_datetime(alarms["timestamp"].str.strip(), format="ISO8601")
alarms["code"] = alarms["code"].astype(int)

print(len(alarms), "alarms")
print(alarms["level"].value_counts())

397 alarms
level
WARNING    216
INFO       180
ERROR        1
Name: count, dtype: int64


## 4. Maintenance book
**EDA findings used:**
- Entries are free text, so we add a **category** (glass / nozzle / other), like the worked example.
- **New decision (not answered in EDA):** some entries have **no time**. Here we give them the start of their shift (early = 06:00, late = 14:00) and mark them `time_estimated = True`, so we never pretend we know the exact time.

In [5]:
maint = pd.read_excel(DATA / "maintenance" / "maintenance_log.xlsx")
SHIFT_START = {"early": "06:00", "late": "14:00"}

maint["time_estimated"] = maint["Time"].isna()
time_text = maint["Time"].fillna(maint["Shift"].map(SHIFT_START)).astype(str).str[:5]
maint["time"] = pd.to_datetime(maint["Date"].dt.strftime("%Y-%m-%d") + " " + time_text)

def category(text):
    t = text.lower()
    if "glass" in t:
        return "glass"
    if "nozzle" in t:
        return "nozzle"
    return "other"

maint["category"] = maint["Entry"].apply(category)
print(maint["category"].value_counts())
print("entries with estimated time:", maint["time_estimated"].sum())

category
glass     28
nozzle     8
other      8
Name: count, dtype: int64
entries with estimated time: 6


## 5. Link each alarm to the job that was running
**EDA finding used:** alarms have **no job ID**, only a time. The rule: an alarm belongs to the job whose `StartTime ≤ alarm time ≤ EndTime`.

**New decision:** an alarm that happens *between* two jobs (for example during a pallet change) gets **no job**. We keep it on the timeline anyway, because it still tells us about the machine.

`merge_asof` finds, for each alarm, the last job that started before it. Then we check that the job hadn't already ended.

In [6]:
alarms = alarms.sort_values("timestamp")
starts = jobs[["JobID", "StartTime", "EndTime"]].sort_values("StartTime")

linked = pd.merge_asof(alarms, starts, left_on="timestamp", right_on="StartTime", direction="backward")
during_job = linked["timestamp"] <= linked["EndTime"]
linked["job_id"] = linked["JobID"].where(during_job)

alarms = linked.drop(columns=["JobID", "StartTime", "EndTime"])
print("alarms during a job :", alarms["job_id"].notna().sum())
print("alarms between jobs :", alarms["job_id"].isna().sum())

alarms during a job : 347
alarms between jobs : 50


## 6. Build the timeline
Every source is turned into the **same five columns**: `time, event_type, source, job_id, detail`. Then all of them are stacked (`concat`) and sorted by time.

**EDA finding used:** INFO alarms are normal operation, so we keep only WARNING and ERROR. That's the same filter you used in Excel.

In [7]:
ev_jobs = pd.DataFrame({
    "time": jobs["StartTime"], "event_type": "job_start", "source": "job_log", "job_id": jobs["JobID"],
    "detail": jobs["ProgramName"] + " | " + jobs["CuttingGas"] + " | nozzle " + jobs["NozzleOperatingHours"].astype(str) + " h",
})

important = alarms[alarms["level"].isin(["WARNING", "ERROR"])]
ev_alarms = pd.DataFrame({
    "time": important["timestamp"], "event_type": "alarm", "source": "alarm_log", "job_id": important["job_id"],
    "detail": important["code"].astype(str) + " " + important["level"] + " | " + important["message"],
})

ev_maint = pd.DataFrame({
    "time": maint["time"], "event_type": "maintenance_" + maint["category"], "source": "maintenance_log", "job_id": None,
    "detail": maint["Entry"] + maint["time_estimated"].map({True: " (time estimated)", False: ""}),
})

ev_insp = pd.DataFrame({
    "time": insp["inspected_at"], "event_type": "inspection", "source": "inspection_records", "job_id": insp["job_id"],
    "detail": insp["result"] + " | " + insp["defect_type"],
})

timeline = (pd.concat([ev_jobs, ev_alarms, ev_maint, ev_insp], ignore_index=True)
              .sort_values("time").reset_index(drop=True))
print(len(timeline), "events")
timeline["event_type"].value_counts()

1204 events


event_type
job_start             527
inspection            416
alarm                 217
maintenance_glass      28
maintenance_nozzle      8
maintenance_other       8
Name: count, dtype: int64

## 7. Read one bad period: bunch B5 (dross, 22–28 Sep)
This is the same story you pieced together across four Excel sheets, now on one screen.
Try changing the dates to look at bunch C (28 Sep – 2 Oct) or D (6 – 8 Oct).

In [8]:
pd.set_option("display.max_colwidth", 90)
window = timeline[(timeline["time"] >= "2026-09-21 12:00") & (timeline["time"] <= "2026-09-28 18:00")]
window[window["event_type"] != "job_start"]      # hide job starts to keep it short

,time,event_type,source,job_id,detail
582,2026-09-21 12:51:28,alarm,alarm_log,JOB-00274,2105 WARNING | Protective glass contamination - check and clean
583,2026-09-21 12:58:19,alarm,alarm_log,JOB-00274,4001 WARNING | Laser power reduced - beam path monitoring
584,2026-09-21 13:04:00,inspection,inspection_records,JOB-00273,NOK | incomplete_cut
586,2026-09-21 13:27:31,alarm,alarm_log,JOB-00275,2105 WARNING | Protective glass contamination - check and clean
587,2026-09-21 13:28:40,alarm,alarm_log,JOB-00275,4001 WARNING | Laser power reduced - beam path monitoring
...,...,...,...,...,...
811,2026-09-28 14:57:20,alarm,alarm_log,JOB-00369,2105 WARNING | Protective glass contamination - check and clean
812,2026-09-28 15:29:00,inspection,inspection_records,JOB-00368,NOK | incomplete_cut
813,2026-09-28 15:37:00,inspection,inspection_records,JOB-00365,NOK | dross
814,2026-09-28 16:44:00,inspection,inspection_records,JOB-00367,NOK | dross


## 8. Check: does the timeline show the same bunches?
**EDA finding used:** normally about 75% of inspected jobs are ok, and a day below about 34% is a real problem.
If this check shows the same bad days as your Excel chart, the cleaning and linking are right.

In [9]:
daily = (insp.assign(date=insp["inspected_at"].dt.date, ok=insp["defect_type"].eq("ok"))
             .groupby("date")["ok"].agg(["mean", "count"]))
daily.columns = ["ok_rate", "inspected"]
print("Days below the 34% limit:")
daily[daily["ok_rate"] < 0.34]

Days below the 34% limit:


,ok_rate,inspected
date,,
2026-09-24,0.266667,15
2026-09-25,0.333333,12
2026-09-26,0.111111,9
2026-09-27,0.000000,1
2026-09-28,0.071429,14
2026-10-04,0.000000,1
2026-10-07,0.133333,15


Note: this groups by **inspection date**, while your Excel chart grouped by **job date**. Inspections can happen hours after the job, which is why a **Sunday** (27 Sep, 4 Oct) can appear with a single late inspection. Which date should you use? That's another decision EDA didn't answer. Job date is usually better, because that's when the defect was made.

## 9. Save
The timeline is saved as a CSV in `data/processed/` (ignored by git). The knowledge graph stage will load this file.

In [10]:
timeline.to_csv(OUT / "event_timeline.csv", index=False)
print("saved", OUT / "event_timeline.csv")

saved ..\data\processed\event_timeline.csv


## What this notebook showed
- Almost every line of cleaning code came **directly from an EDA finding**: separators, decimals, ID typos, the mapping, INFO vs WARNING.
- Two decisions only appeared **while building**: alarms between jobs, and maintenance without a time. They're written down in the notebook so you can justify them in the report.
- The check in step 8 uses your EDA results to **test** the code.

**Next:** turn this timeline into nodes and links for the knowledge graph.